In [0]:
from pyspark.sql import functions as F

S = "workspace.home_credit_silver"
c = spark.table(f"{S}.customer")
overall = c.agg(F.avg("target")).first()[0]
print("overall default rate:", round(overall, 4))

def test(buckets):
    """buckets = a DataFrame with sk_id_curr and a 'bucket' column"""
    (c.join(buckets, "sk_id_curr", "left")
       .fillna({"bucket": "0_no history"})
       .groupBy("bucket")
       .agg(F.count("*").alias("customers"),
            F.round(F.avg("target"), 4).alias("default_rate"))
       .withColumn("lift", F.round(F.col("default_rate") / overall, 2))
       .orderBy("bucket").show())

In [0]:
p = spark.table(f"{S}.previous_application")
refused = p.groupBy("sk_id_curr").agg(F.sum(F.col("is_refused").cast("int")).alias("n"))
test(refused.select("sk_id_curr", F.when(F.col("n") == 0, "1_zero refusals")
                                   .when(F.col("n") <= 2, "2_1-2 refusals")
                                   .otherwise("3_3+ refusals").alias("bucket")))

In [0]:
b = spark.table(f"{S}.bureau_loan")
active = b.groupBy("sk_id_curr").agg(F.sum(F.col("is_active").cast("int")).alias("n"))
test(active.select("sk_id_curr", F.when(F.col("n") == 0, "1_no active loans")
                                  .when(F.col("n") <= 2, "2_1-2 active")
                                  .otherwise("3_3+ active").alias("bucket")))

In [0]:
b = spark.table(f"{S}.bureau_loan")
active = b.groupBy("sk_id_curr").agg(F.sum(F.col("is_active").cast("int")).alias("n"))
active.limit(5).display()

In [0]:
%sql
select count(*), 
    count(distinct sk_id_curr) 
from workspace.home_credit_gold.customer_risk_profile;

In [0]:
%sql
SELECT * FROM workspace.home_credit_gold.segment_risk_summary ORDER BY lift;